# Incremental Learning
# 增量学习

PipelineTS supports **incremental learning** via the `update()` method — update trained models with new data without retraining from scratch.
PipelineTS 通过 `update()` 方法支持**增量学习** — 使用新数据更新已训练模型，无需从头重新训练。

This tutorial covers:
本教程涵盖：

1. **Why incremental learning / 为什么需要增量学习**
2. **ModelPipeline `update()` / 管道增量更新**
3. **SmartRouter `update()` / 智能路由器增量更新**
4. **How different model types handle updates / 不同模型类型的更新策略**
5. **Simulating streaming data / 模拟流式数据**
6. **Best practices / 最佳实践**

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
LAGS = 12

## 1. Why Incremental Learning
## 1. 为什么需要增量学习

In production, new data arrives continuously. Retraining from scratch every time is:
在生产环境中，新数据持续到达。每次从头重新训练：

- **Expensive**: Full retraining costs grow with data size / **昂贵**：完全重训练成本随数据量增长
- **Slow**: May not keep up with data arrival rate / **缓慢**：可能跟不上数据到达速率
- **Wasteful**: Previous training effort is discarded / **浪费**：之前的训练努力被丢弃

Incremental learning solves this by **updating** existing models with new data.
增量学习通过用新数据**更新**现有模型来解决此问题。

## 2. Prepare Data / 准备数据

We split data into initial training set and simulated "new" batches.

我们将数据分为初始训练集和模拟的"新"批次。

In [ ]:
# Generate 300 days of data / 生成 300 天数据
n = 300
dates = pd.date_range(start='2020-01-01', periods=n, freq='D')
values = 50 + 10 * np.sin(np.linspace(0, 8 * np.pi, n)) + np.random.randn(n) * 2
full_data = pd.DataFrame({'date': dates, 'value': values})

# Split: initial 200 for training, then 3 batches of ~33 each
# 分割：前 200 条训练，之后 3 批每批约 33 条
initial_data = full_data.iloc[:200].copy()
batch_1 = full_data.iloc[200:233].copy()
batch_2 = full_data.iloc[233:266].copy()
batch_3 = full_data.iloc[266:300].copy()

print(f"Initial training data / 初始训练数据: {len(initial_data)} rows")
print(f"Batch 1 / 第 1 批: {len(batch_1)} rows")
print(f"Batch 2 / 第 2 批: {len(batch_2)} rows")
print(f"Batch 3 / 第 3 批: {len(batch_3)} rows")

## 3. ModelPipeline `update()`
## 3. ModelPipeline 增量更新

In [ ]:
from PipelineTS.pipeline import ModelPipeline

# Step 1: Initial training / 第 1 步：初始训练
pipeline = ModelPipeline(
    time_col='date', target_col='value', lags=LAGS,
    include_models=['torch_boosting_forest', 'torch_bagging_forest'],
    quantile=0.9, cv=2,
)
pipeline.fit(initial_data)

pred_initial = pipeline.predict(10)
print("Initial prediction / 初始预测:")
pred_initial.head()

In [ ]:
# Step 2: Update with batch 1 / 第 2 步：用第 1 批更新
pipeline.update(batch_1)

pred_after_b1 = pipeline.predict(10)
print("After batch 1 update / 第 1 批更新后:")
pred_after_b1.head()

In [ ]:
# Step 3: Update with batch 2 / 第 3 步：用第 2 批更新
pipeline.update(batch_2)

pred_after_b2 = pipeline.predict(10)
print("After batch 2 update / 第 2 批更新后:")
pred_after_b2.head()

In [ ]:
# Visualize how forecasts improve with more data
# 可视化预测如何随数据量增加而改善
from PipelineTS.plot import plot_model_comparison

plot_model_comparison(
    full_data, {
        'Initial (200 pts)': pred_initial,
        'After Batch 1 (233 pts)': pred_after_b1,
        'After Batch 2 (266 pts)': pred_after_b2,
    },
    time_col='date', target_col='value',
    history_tail=50,
    title='增量学习：预测随数据更新的变化',
    lang='zh',
)

## 4. SmartRouter `update()`
## 4. SmartRouter 增量更新

SmartRouter also supports `update()`, which delegates to its internal ModelPipeline.

SmartRouter 也支持 `update()`，委托给内部 ModelPipeline。

In [ ]:
from PipelineTS.pipeline import SmartRouter

router = SmartRouter(
    time_col='date', target_col='value',
    max_models=3,
)
router.fit(initial_data)

print("Initial SmartRouter prediction / 初始 SmartRouter 预测:")
router.predict(5).head()

In [ ]:
# Incremental update / 增量更新
router.update(batch_1)

print("After update / 更新后:")
router.predict(5).head()

## 5. How Different Model Types Handle Updates
## 5. 不同模型类型的更新策略

| Model Type / 模型类型 | Strategy / 策略 | Details / 详情 |
|---|---|---|
| **Neural Networks** / 神经网络 | Warm-start / 热启动 | Continue training with fewer epochs on combined data / 在合并数据上以更少轮次继续训练 |
| **GBDT** (TorchBoostingForest, TorchBaggingForest) | Refit / 重新拟合 | Full refit on combined old + new data / 在合并数据上完全重新拟合 |
| **Prophet** | Refit / 重新拟合 | Refit on combined data / 在合并数据上重新拟合 |
| **AutoARIMA** | Refit / 重新拟合 | Re-search on combined data / 在合并数据上重新搜索 |

### NN Warm-Start Example / 神经网络热启动示例

In [ ]:
nn_pipeline = ModelPipeline(
    time_col='date', target_col='value', lags=LAGS,
    include_models=['n_linear', 'd_linear'],
    quantile=0.9, cv=2,
    n_linear__epochs=50,
    n_linear__patience=10,
    n_linear__verbose=False,
    d_linear__epochs=50,
    d_linear__patience=10,
    d_linear__verbose=False,
)
nn_pipeline.fit(initial_data)

print("NN leaderboard before update / 更新前 NN 排行榜:")
print(nn_pipeline.leader_board_[['model', 'metric']])

# Update with new batch / 用新批次更新
nn_pipeline.update(batch_1)

print("\nNN prediction after update / 更新后 NN 预测:")
nn_pipeline.predict(5).head()

## 6. Simulating Streaming Data
## 6. 模拟流式数据

In a real scenario, you would call `update()` whenever new data arrives.

在真实场景中，每当新数据到达时调用 `update()`。

In [ ]:
# Simulate streaming: initial fit, then 3 incremental updates
# 模拟流式数据：初始训练，然后 3 次增量更新

stream_pipeline = ModelPipeline(
    time_col='date', target_col='value', lags=LAGS,
    include_models=['torch_boosting_forest'],
    quantile=None, cv=2,
)

# Initial fit / 初始训练
stream_pipeline.fit(initial_data)
print(f"[Initial] Trained on {len(initial_data)} rows")
print(f"  Prediction: {stream_pipeline.predict(3)['value'].values}")

# Streaming updates / 流式更新
batches = [batch_1, batch_2, batch_3]
total = len(initial_data)
for i, batch in enumerate(batches, 1):
    stream_pipeline.update(batch)
    total += len(batch)
    pred = stream_pipeline.predict(3)
    print(f"[Batch {i}] Updated to {total} rows")
    print(f"  Prediction: {pred['value'].values}")

## 7. Error Handling / 错误处理

`update()` raises `ValueError` if the pipeline has not been fitted yet.

`update()` 在管道尚未拟合时抛出 `ValueError`。

In [ ]:
# This will raise ValueError / 这将抛出 ValueError
try:
    unfitted = ModelPipeline(
        time_col='date', target_col='value', lags=LAGS,
        include_models=['torch_boosting_forest'],
    )
    unfitted.update(batch_1)
except ValueError as e:
    print(f"Expected error / 预期错误: {e}")

## 8. Best Practices / 最佳实践

1. **Batch size**: Update with reasonably-sized batches (not single observations) for efficiency.
1. **批次大小**：使用合理大小的批次更新（不是单个观测值），以提高效率。

2. **Periodic full retraining**: Occasionally do a full `fit()` to recalibrate from scratch.
2. **定期完全重训练**：偶尔执行完全 `fit()` 以从头重新校准。

3. **Monitor drift**: Track prediction quality over time; if it degrades, do a full retrain.
3. **监控漂移**：随时间跟踪预测质量；如果退化，进行完全重训练。

4. **Save checkpoints**: Save the pipeline after important updates using `save_model()`.
4. **保存检查点**：使用 `save_model()` 在重要更新后保存管道。

In [ ]:
from PipelineTS.io import save_model

# Save after incremental update / 增量更新后保存
# save_model('pipeline_updated.zip', stream_pipeline)
print("Uncomment the line above to save. / 取消注释上面的行即可保存。")

## Summary / 总结

| Feature / 功能 | API | Description / 描述 |
|---|---|---|
| Pipeline update / 管道更新 | `pipeline.update(new_data)` | Update all models with new data / 用新数据更新所有模型 |
| SmartRouter update / 路由器更新 | `router.update(new_data)` | Delegates to internal pipeline / 委托给内部管道 |
| NN warm-start / NN 热启动 | Automatic / 自动 | Fewer epochs on combined data / 在合并数据上使用更少轮次 |
| GBDT/Stat refit / GBDT/统计重训练 | Automatic / 自动 | Full refit on combined data / 在合并数据上完全重训练 |
| Error guard / 错误保护 | `ValueError` | Raises if not fitted / 未拟合时抛出 |